## Named Entity Recognition


In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding
import random
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

MODEL_NAME = "en_core_web_md"
OUTPUT_MODEL = "ner_model"
TRAIN_DATA_PATH = "train.spacy"
JSON_DATA_PATH = "ner_train_spacy.json"

print(f"Loading base model '{MODEL_NAME}'...")
nlp = spacy.load(MODEL_NAME)

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    ner = nlp.get_pipe("ner")

labels = ["PARTY", "AGREEMENT", "SECTION", "EXHIBIT", "DATE", "POLICY"]
for label in labels:
    ner.add_label(label)

print(f"Schema initialized with labels: {labels}")

def convert_to_docbin():
    from spacy.tokens import DocBin
    db = DocBin()
    
    if not Path(JSON_DATA_PATH).exists():
        print(f"Error: {JSON_DATA_PATH} not found.")
        return
        
    with open(JSON_DATA_PATH, "r", encoding="utf-8") as f:
        training_data = json.load(f)
    
    print(f"Converting {len(training_data)} examples for your custom schema...")
    skipped = 0
    for item in training_data:
        text = item.get("text") or item.get("clause")
        entities = item.get("entities", [])
        
        doc = nlp.make_doc(text)
        ents = []
        for ent_data in entities:
            if isinstance(ent_data, list):
                start, end, label = ent_data
            else:
                start, end, label = ent_data["start"], ent_data["end"], ent_data["label"]
            if label in labels:
                span = doc.char_span(start, end, label=label, alignment_mode="expand")
                if span is not None:
                    ents.append(span)
                else:
                    skipped += 1
        
        try:
            doc.ents = ents
            db.add(doc)
        except Exception:
            skipped += 1
            
    db.to_disk(TRAIN_DATA_PATH)
    print(f"Saved training binary to {TRAIN_DATA_PATH}. (Skipped {skipped} misaligned spans)")

convert_to_docbin()

doc_bin = spacy.tokens.DocBin().from_disk(TRAIN_DATA_PATH)
examples = []
for doc in doc_bin.get_docs(nlp.vocab):
    examples.append(Example.from_dict(doc, {"entities": [(e.start_char, e.end_char, e.label_) for e in doc.ents]}))

optimizer = nlp.resume_training()

unaffected_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]

EPOCHS = 40
print(f"Starting fine-tuning for {EPOCHS} epochs...")

with nlp.disable_pipes(*unaffected_pipes):
    for epoch in range(EPOCHS):
        random.shuffle(examples)
        losses = {}
        batches = minibatch(examples, size=compounding(4.0, 32.0, 1.001))
        for batch in batches:
            nlp.update(
                batch,
                drop=0.3,
                losses=losses,
                sgd=optimizer
            )
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:2d}/{EPOCHS} | NER Loss: {losses.get('ner', 0):.6f}")

nlp.to_disk(OUTPUT_MODEL)
print(f"Success! Corrected model saved to '{OUTPUT_MODEL}'")

# Load trained model
nlp = spacy.load("ner_model")
print("Loaded trained model")

# Read clauses
clauses_path = "output/SPONSORSHIP_AGREEMENT_clauses.txt"
with open(clauses_path, "r", encoding="utf-8") as f:
    clauses = [line.strip() for line in f if line.strip()]

print(f"Processing {len(clauses)} clauses...")

results = []

for clause in clauses:
    doc = nlp(clause)
    entities = []
    
    for ent in doc.ents:
        entities.append({
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })
    
    results.append({
        "clause": clause,
        "entities": entities
    })

# Save results
output_path = "output/ner_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Inference completed! Results saved to '{output_path}'")
print(f"Total entities extracted: {sum(len(r['entities']) for r in results)}")

# Clause Intent Classification


In [ ]:
import re
import json
from pathlib import Path
from collections import Counter

clauses_path = Path("output/SPONSORSHIP_AGREEMENT_clauses.txt")

if not clauses_path.exists():
    clauses_path = Path("output/clauses.txt")

with open(clauses_path, "r", encoding="utf-8") as f:
    clauses = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(clauses)} clauses from {clauses_path.name}")

def classify_intent_detailed(clause_text):
    text_lower = clause_text.lower()
    if re.search(r"^(exhibit|section|article|schedule)\b", text_lower):
        return "Scope/Structure", "struct_keyword"

    rules = [
        (r"\b(shall mean|is defined as|means|meaning provided in|shall have the meaning|will.*?be referred to as|shall.*?be referred to as)\b", "Definition", "def_keyword"),
        (r"\b(shall not|may not|must not|in no event shall|prohibited from)\b", "Prohibition", "prohib_keyword"),
        (r"\b(if|provided that|provided,? however|in the event|subject to|unless|notwithstanding)\b", "Condition", "cond_keyword"),
        (r"\b(may|has the right to|entitled to|at (its|their) discretion|reserves the right)\b", "Permission", "perm_keyword"),
        (r"\b(represents|warrants|acknowledges|disclaims|agrees that|recognizes)\b", "Representation/Warranty", "rep_keyword"),
        (r"\b(shall|must|agrees to|will|is responsible for|required to)\b", "Obligation", "oblig_keyword")
    ]

    best_match = ("Other", "no_match")
    earliest_index = float('inf')

    for pattern, intent, rule_name in rules:
        match = re.search(pattern, text_lower)
        if match:
            if match.start() < earliest_index:
                earliest_index = match.start()
                best_match = (intent, rule_name)
    return best_match

results = []
for i, clause in enumerate(clauses, 1):
    clause = clause.strip()
    if not clause:
        continue
    intent, triggered_rule = classify_intent_detailed(clause)
    results.append({
        "clause_id": i,
        "clause": clause,
        "intent": intent,
        "trigger_rule": triggered_rule
    })

intent_counts = Counter([r["intent"] for r in results])
print("Intent Distribution:")
for intent, count in sorted(intent_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{intent:25} | {count}")

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "intent_classification.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

with open(output_dir / "intent_classification.txt", "w", encoding="utf-8") as f:
    for r in results:
        f.write(f"{r['intent']:25} | Rule: {r['trigger_rule']:15} | {r['clause'][:100].replace(chr(10), ' ')}...\n")

print("\nFiles successfully saved with detailed intents and trigger rules!")

Loaded 539 clauses from SPONSORSHIP_AGREEMENT_clauses.txt
Intent Distribution:
Other                     | 226
Obligation                | 157
Condition                 | 54
Definition                | 47
Permission                | 23
Scope/Structure           | 17
Prohibition               | 10
Representation/Warranty   | 5

Files successfully saved with detailed intents and trigger rules!
